# Mini-Projeto ETL — IQVIA Data
## Notebook de Simulação — Mudanças para Teste do SCD Tipo 2

**Objetivo:** Simular eventos reais que acionam o SCD Tipo 2 na camada Gold:

| Evento | Produto (EAN) | Descrição |
|--------|---------------|-----------|
| ✏️ Alteração de valor | `42360414` | Valor alterado de `210` → `999` |
| 🗑️ Exclusão lógica | `42176763` | Produto descontinuado — removido do novo arquivo |

### O que este notebook faz
1. Lê o arquivo Silver existente (`vendas_enriquecido.parquet`)
2. Aplica as modificações simuladas
3. Salva um **novo arquivo Silver** com sufixo `_v2` na mesma partição
4. Este novo arquivo será consumido pelo notebook Gold (`03_load_gold_scd.ipynb`) para demonstrar o SCD2 em ação

> ⚠️ Este notebook é apenas para fins de demonstração do SCD Tipo 2. Em produção, os dados viriam do pipeline Bronze → Silver normalmente.

## 1. Instalação e Importação de Dependências

In [1]:
%pip install -r requirements.txt --quiet

Note: you may need to restart the kernel to use updated packages.


ERROR: Cannot install db-dtypes==1.5.0 and pandas==3.0.1 because these package versions have conflicting dependencies.

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-with-dependency-conflicts


In [2]:
import io
from datetime import datetime, timezone

import pandas as pd
from google.cloud import storage

## 2. Configurações

In [3]:
GCS_PROJECT_ID    = "proc-de-dados-iqvia-com-scd"
GCS_BUCKET_NAME   = "etl-iqvia-data-lake-augusto"
GCS_SILVER_PREFIX = "silver/iqvia"

# Partição Silver original (onde os dados da carga anterior estão)
SILVER_DATE_PARTITION = "2026/03/20"

# Arquivo de origem e destino da simulação
SOURCE_BLOB = f"{GCS_SILVER_PREFIX}/{SILVER_DATE_PARTITION}/vendas_enriquecido.parquet"
TARGET_BLOB = f"{GCS_SILVER_PREFIX}/{SILVER_DATE_PARTITION}/vendas_enriquecido_v2.parquet"

LOAD_TIMESTAMP = datetime.now(timezone.utc)

print(f"Projeto GCP   : {GCS_PROJECT_ID}")
print(f"Origem        : gs://{GCS_BUCKET_NAME}/{SOURCE_BLOB}")
print(f"Destino (sim) : gs://{GCS_BUCKET_NAME}/{TARGET_BLOB}")

Projeto GCP   : proc-de-dados-iqvia-com-scd
Origem        : gs://etl-iqvia-data-lake-augusto/silver/iqvia/2026/03/20/vendas_enriquecido.parquet
Destino (sim) : gs://etl-iqvia-data-lake-augusto/silver/iqvia/2026/03/20/vendas_enriquecido_v2.parquet


## 3. Conexão com o GCS

In [4]:
# Autenticação via Application Default Credentials (ADC).
# Execute previamente no terminal (requer Google Cloud SDK instalado):
#   gcloud auth application-default login
#   gcloud auth application-default set-quota-project proc-de-dados-iqvia-com-scd

gcs_client = storage.Client(project=GCS_PROJECT_ID)
bucket     = gcs_client.bucket(GCS_BUCKET_NAME)

print(f"Conectado ao GCS | Projeto: {gcs_client.project}")

Conectado ao GCS | Projeto: proc-de-dados-iqvia-com-scd


## 4. Leitura do Silver Original

In [5]:
blob = bucket.blob(SOURCE_BLOB)
df_original = pd.read_parquet(io.BytesIO(blob.download_as_bytes()))

print(f"Shape original: {df_original.shape}")

# Exibe os produtos únicos com seus valores consolidados
df_resumo = (
    df_original
    .groupby(["ean", "cod_prod_catarinense"], as_index=False)
    .agg(valor_produto=("vol_so_preco_popular", "sum"))
    .sort_values("ean")
)
print("\nProdutos únicos (antes das modificações):")
display(df_resumo)

Shape original: (45, 11)

Produtos únicos (antes das modificações):


,ean,cod_prod_catarinense,valor_produto
0,32689150,741013,25
1,42110200,630693,25
2,42176763,687607,0
3,42277217,739189,115
4,42355014,735367,5
5,42355465,735344,0
6,42360407,734398,75
7,42360414,734399,210
8,42389248,736535,35


## 5. Aplicando as Modificações Simuladas

### Evento 1 — ✏️ Alteração de Valor
**EAN `42360414`**: `vol_so_preco_popular` alterado de `210` → `999`  
*(Simula uma correção de valor de produto)*

In [6]:
EAN_ALTERADO = 42360414
NOVO_VALOR   = 999

df_modificado = df_original.copy()

mask_alterado = df_modificado["ean"] == EAN_ALTERADO
linhas_afetadas = mask_alterado.sum()

df_modificado.loc[mask_alterado, "vol_so_preco_popular"] = NOVO_VALOR
df_modificado.loc[mask_alterado, "dt_processamento"]     = LOAD_TIMESTAMP.strftime("%d/%m/%Y %H:%M:%S UTC")

print(f"EAN {EAN_ALTERADO}: {linhas_afetadas} linha(s) com vol_so_preco_popular → {NOVO_VALOR}")
display(df_modificado[df_modificado["ean"] == EAN_ALTERADO][["ean", "cod_prod_catarinense", "vol_so_preco_popular", "dt_processamento"]])

EAN 42360414: 5 linha(s) com vol_so_preco_popular → 999


,ean,cod_prod_catarinense,vol_so_preco_popular,dt_processamento
35,42360414,734399,999,20/03/2026 12:57:51 UTC
36,42360414,734399,999,20/03/2026 12:57:51 UTC
37,42360414,734399,999,20/03/2026 12:57:51 UTC
38,42360414,734399,999,20/03/2026 12:57:51 UTC
39,42360414,734399,999,20/03/2026 12:57:51 UTC


### Evento 2 — 🗑️ Exclusão Lógica
**EAN `42176763`**: produto removido do arquivo (descontinuado)  
*(Simula um produto que saiu do mix — ausente na próxima carga)*

In [7]:
EAN_REMOVIDO = 42176763

linhas_antes  = len(df_modificado)
df_modificado = df_modificado[df_modificado["ean"] != EAN_REMOVIDO].copy()
linhas_depois = len(df_modificado)

print(f"EAN {EAN_REMOVIDO}: {linhas_antes - linhas_depois} linha(s) removida(s) do arquivo")
print(f"Shape após remoção: {df_modificado.shape}")

EAN 42176763: 5 linha(s) removida(s) do arquivo
Shape após remoção: (40, 11)


## 6. Comparativo Antes × Depois

In [8]:
df_antes = (
    df_original
    .groupby(["ean", "cod_prod_catarinense"], as_index=False)
    .agg(valor_antes=("vol_so_preco_popular", "sum"))
)

df_depois = (
    df_modificado
    .groupby(["ean", "cod_prod_catarinense"], as_index=False)
    .agg(valor_depois=("vol_so_preco_popular", "sum"))
)

df_comparativo = df_antes.merge(df_depois, on=["ean", "cod_prod_catarinense"], how="outer")
df_comparativo["situacao"] = df_comparativo.apply(
    lambda r: "🗑️ REMOVIDO" if pd.isna(r["valor_depois"])
    else ("✏️ ALTERADO" if r["valor_antes"] != r["valor_depois"]
    else "✅ SEM ALTERAÇÃO"),
    axis=1
)

print("Comparativo de produtos — antes e depois das modificações:")
display(df_comparativo.sort_values("ean"))

Comparativo de produtos — antes e depois das modificações:


,ean,cod_prod_catarinense,valor_antes,valor_depois,situacao
0,32689150,741013,25,25,✅ SEM ALTERAÇÃO
1,42110200,630693,25,25,✅ SEM ALTERAÇÃO
2,42176763,687607,0,<NA>,🗑️ REMOVIDO
3,42277217,739189,115,115,✅ SEM ALTERAÇÃO
4,42355014,735367,5,5,✅ SEM ALTERAÇÃO
5,42355465,735344,0,0,✅ SEM ALTERAÇÃO
6,42360407,734398,75,75,✅ SEM ALTERAÇÃO
7,42360414,734399,210,4995,✏️ ALTERADO
8,42389248,736535,35,35,✅ SEM ALTERAÇÃO


## 7. Salvando o Arquivo Simulado no GCS

In [9]:
buffer = io.BytesIO()
df_modificado.to_parquet(buffer, index=False, engine="pyarrow")
buffer.seek(0)

blob_dest = bucket.blob(TARGET_BLOB)
blob_dest.metadata = {
    "layer": "silver",
    "pipeline": "iqvia-etl-simulation",
    "load_timestamp": LOAD_TIMESTAMP.isoformat(),
    "rows": str(len(df_modificado)),
    "note": "Arquivo simulado para teste SCD2: 1 valor alterado, 1 produto removido",
}
blob_dest.upload_from_file(buffer, content_type="application/octet-stream")

size_kb = round(buffer.getbuffer().nbytes / 1024, 2)
print(f"Arquivo salvo: gs://{GCS_BUCKET_NAME}/{TARGET_BLOB}")
print(f"Tamanho: {size_kb} KB | Linhas: {len(df_modificado)}")

Arquivo salvo: gs://etl-iqvia-data-lake-augusto/silver/iqvia/2026/03/20/vendas_enriquecido_v2.parquet
Tamanho: 7.49 KB | Linhas: 40


## 8. Próximo Passo — Rodar o Pipeline Gold com os Dados Simulados

Abra o notebook `03_load_gold_scd.ipynb` e altere a configuração do arquivo Silver para apontar para a versão simulada:

```python
# Em 03_load_gold_scd.ipynb — célula de configurações:
SILVER_DATE_PARTITION = "2026/03/20"
SILVER_FILE_NAME      = "vendas_enriquecido_v2.parquet"  # arquivo simulado
```

### O que esperar no resultado Gold:

| Produto (EAN) | Evento | Ação SCD2 esperada |
|---------------|--------|--------------------|
| `42360414` | ✏️ Valor `210` → `999` | Fecha registro antigo (`flag_ativo=FALSE`) + insere novo ativo |
| `42176763` | 🗑️ Ausente no novo arquivo | Fecha registro (`flag_ativo=FALSE`, descontinuado) |
| Demais 7 EANs | ✅ Sem alteração | Nenhuma ação |